In [1]:
!pip install clu
!pip install tensorflow-cpu

: 

In [ ]:
import os

# 1. CRITICAL: SET ENVIRONMENT FLAGS BEFORE IMPORTING JAX
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "False"  # TPU does not support float64 natively

import jax
import jax.numpy as jnp

# 2. CONNECT TO COLAB TPU
try:
    # Modern Colab TPU VM way
    jax.distributed.initialize()
except Exception:
    try:
        # Legacy Colab TPU Node way
        import jax.tools.colab_tpu
        jax.tools.colab_tpu.setup_tpu()
    except:
        print("TPU initialization failed, check Runtime Type.")

print(f"TPU Cores: {jax.device_count()}")
print(f"Default Backend: {jax.default_backend()}") # Should be 'tpu'

from collections.abc import Iterable, Sequence
from typing import Any

import os
from clu import metric_writers
import optax
import sys

sys.path.append("./")

import train.callbacks as cb
from train import trainers
import utils
from data.beltramiDataGeneration import *
from model.unet3d import UNet3d
from train.models import DataAssimilationModel
from train.trainers import BeltramiTrainer

print(f"Devices found: {jax.devices()}")

def run(
    *,
    train_dataloader: Iterable[Any],
    trainer: trainers.BaseTrainer,
    workdir: str,
    # training configs
    total_train_steps: int,
    metric_aggregation_steps: int = 50,
    # evaluation configs
    eval_dataloader: Iterable[Any] | None = None,
    eval_every_steps: int = 100,
    num_batches_per_eval: int = 10,
    run_sanity_eval_batch: bool = True,
    # other configs
    metric_writer: metric_writers.MultiWriter | None = None,
    callbacks: Sequence[cb.Callback] = (),
) -> None:
  """Runs trainer for a training task.

  This function runs a trainer in batches of "metric aggregation" steps, where
  the step-wise metrics obtained within the same batch are aggregated
  (i.e. by computing the average and/or std based on the metric defined in
  the trainer class). The aggregated metrics are then automatically saved to a
  tensorflow event file in `workdir`. Evaluation runs periodically, i.e. once
  every `eval_every_steps` steps, if an eval dataloader is provided.

  Args:
    train_dataloader: A dataloader emitting training data in batches.
    trainer: A trainer object hosting the train and eval logic.
    workdir: The working directory where results (e.g. train & eval metrics) and
      progress (e.g. checkpoints) are saved.
    total_train_steps: Total number of training steps to run.
    metric_aggregation_steps: The trainer runs this number of steps at a time,
      after which training metrics are aggregated and logged.
    eval_dataloader: An evaluation dataloader (optional). If set to `None`, no
      evaluation will run.
    eval_every_steps: The period, in number of train steps, at which evaluation
      runs. Must be an integer multiple of `metric_aggregation_steps`.
    num_batches_per_eval: The number of batches to step through every time
      evaluation is run (resulting metrics are aggregated).
    run_sanity_eval_batch: Whether to step through sanity check eval batch
      before training starts. This helps expose runtime issues early, without
      having to wait until evaluation is first triggered (i.e. after
      `eval_every_steps`).
    metric_writer: A metric writer that writes scalar metrics to disc. It is
      also accessible to callbacks for custom writing in other formats.
    callbacks: A sequence of self-contained programs executing non-essential
      logic (e.g. checkpoint saving, logging, timing, profiling etc.).
  """
  if not os.path.exists(workdir):
    os.makedirs(workdir)

  train_iter = iter(train_dataloader)
  eval_iter = None
  run_evaluation = eval_dataloader is not None
  if run_evaluation:
    if eval_every_steps % metric_aggregation_steps != 0:
      raise ValueError(
          f"`eval_every_steps` ({eval_every_steps}) "
          "must be an integer multiple of "
          "`metric_aggregation_steps` ({metric_aggregation_steps})"
      )
    eval_iter = iter(eval_dataloader)
    if run_sanity_eval_batch:
      trainer.eval(eval_iter, num_steps=1)

  if metric_writer is None:
    metric_writer = metric_writers.create_default_writer(
        workdir, just_logging=jax.process_index() > 0
    )

  for callback in callbacks:
    callback.metric_writer = metric_writer
    callback.on_train_begin(trainer)

  cur_step = trainer.train_state.int_step
  while cur_step < total_train_steps:
    for callback in callbacks:
      callback.on_train_batches_begin(trainer)

    num_steps = min(total_train_steps - cur_step, metric_aggregation_steps)
    train_metrics = trainer.train(train_iter, num_steps).compute()
    cur_step += num_steps
    metric_writer.write_scalars(cur_step, train_metrics)

    # At train/eval batch end, callbacks are called in reverse order so that
    # they are last-in-first-out, loosely resembling nested python contexts.
    for callback in reversed(callbacks):
      callback.on_train_batches_end(trainer, train_metrics)

    if run_evaluation:
      if cur_step == total_train_steps or cur_step % eval_every_steps == 0:
        for callback in callbacks:
          callback.on_eval_batches_begin(trainer)

        assert eval_iter is not None
        eval_metrics = trainer.eval(eval_iter, num_batches_per_eval).compute()
        eval_metrics_to_log = {
            k: v for k, v in eval_metrics.items() if utils.is_scalar(v)
        }
        metric_writer.write_scalars(cur_step, eval_metrics_to_log)

        for callback in reversed(callbacks):
          callback.on_eval_batches_end(trainer, eval_metrics)

  for callback in reversed(callbacks):
    callback.on_train_end(trainer)

  metric_writer.flush()

if __name__=="__main__":
    
  # 1. Generate the Dataset
  # Using spatial_res=32 to fit in memory easily
  # gt, obs, coords, times = generate_da_data(spatial_res=32, time_steps=100)
  gt, obs, coords, times = generate_da_data(spatial_res=8, time_steps=10)

  # 1. Setup Architecture
  unet_config = UNet3d(
      out_channels=4,      # Required: [u, v, w, p]
      dtype=jnp.bfloat16, # Enabled for TPU
      time_cond=True,
      num_channels=(32, 64, 128, 256),
      use_spatial_attention=(False, False, True, True)
  )

  # 2. Update the Model Wrapper accordingly
  da_model = DataAssimilationModel(
      model=unet_config, 
      input_shape=(4, 8, 8, 8, 4) 
  )

  # 4. Define Optimizer & Trainer
  tx = optax.adamw(learning_rate=1e-4, weight_decay=1e-5)
  trainer = BeltramiTrainer(
      model=da_model, 
      optimizer=tx, 
      rng=jax.random.PRNGKey(0)
  )

  # 5. Define Callbacks
  base_workdir = "./beltrami_experiment"
  workdir = os.path.abspath(base_workdir)  # Converts to absolute path
  callbacks = [
      cb.TqdmProgressBar(
          total_train_steps=5000, 
          train_monitors=["train_loss"]
      ),
      cb.TrainStateCheckpoint(base_dir=workdir)
  ]

  # 6. Start Training
  run(
      train_dataloader=beltrami_dataloader(obs, times, batch_size=4),
      trainer=trainer,
      workdir=workdir,
      total_train_steps=5000,
      metric_aggregation_steps=50,
      callbacks=callbacks
  )
  
  # run(
  #     train_dataloader=beltrami_da_dataloader(obs, times, batch_size=4, mode="train"),
  #     trainer=trainer,
  #     workdir=workdir,
  #     total_train_steps=5000,
  #     metric_aggregation_steps=50,
  #     callbacks=callbacks
  # )

# Extra

In [ ]:
import jax
import jax.numpy as jnp
from flax.core import FrozenDict

from model.unet3d import UNet3d

key = jax.random.PRNGKey(0)

batch = 2
nx, ny, nz = 16, 16, 16   # 3D grid
channels = 3               # e.g., velocity components

x = jax.random.normal(key, (batch, nx, ny, nz, channels))
sigma = jnp.ones((batch,)) * 0.5
time = jnp.linspace(0.0, 1.0, batch)

model = UNet3d(
    out_channels=channels,
    time_cond=True,
    num_channels = (16, 32),
    downsample_ratio = (2, 2),
    num_blocks=2,
    noise_embed_dim=32,
    input_proj_channels=16,
    output_proj_channels=16,
    padding="SAME",
    use_spatial_attention=(False, True),
)

variables = model.init(
    key,
    x,
    sigma,
    time,
    is_training=True,
)

params = variables["params"]
print("Parameter shapes:")
jax.tree_util.tree_map(lambda a: a.shape, params)

y = model.apply(
    variables,
    x,
    sigma,
    time,
    is_training=False,
)

print("Output shape:", y.shape)

In [ ]:
from data.beltramiDataGeneration import generate_da_data
# import matplotlib.pyplot as plt

# # Parameters
# A_PARAM = 1.0
# D_PARAM = 1.0
# SPATIAL_RES = 16  # Reduced for quick demonstration
# TIME_STEPS = 20
# DT = 0.05

# # Generate Data
# truth, obs, coords, t_array = generate_da_data(
#     spatial_res=SPATIAL_RES,
#     time_steps=TIME_STEPS,
#     dt=DT,
#     a=A_PARAM,
#     d=D_PARAM
# )

# # Shapes
# print("\n--- Data Generation Complete ---")
# print(f"Ground Truth Shape: {truth.shape} (Time, X, Y, Z, Vars)")
# print(f"Variables channel index: 0:u, 1:v, 2:w, 3:p")

# # --- Visualization (Optional Check) ---
# # Plotting the U-velocity field at the middle Z-slice at t=0
# z_idx = SPATIAL_RES // 2
# u_field_t0 = truth[0, :, :, z_idx, 0] # Time 0, All X, All Y, Mid Z, Var U

# plt.figure(figsize=(6, 5))
# plt.contourf(coords['x'], coords['y'], u_field_t0, levels=50, cmap='viridis')
# plt.colorbar(label='u velocity')
# plt.title(f'Beltrami Flow: U-Velocity (z=0, t=0)\na={A_PARAM}, d={D_PARAM}')
# plt.xlabel('x')
# plt.ylabel('y')
# plt.show()